# colony64

> Phase 18, a second user-approved deviation (`specs/multiagent/phase18/`): guards at 64 agents. Guards beat the plain board at 8, 16 and 32 agents with no trend over size; this run adds 64 agents in a lean form: the plain board and guards with independent liars only (no clean boards), the Stage 1 seeds, under a cap the user raised to $250. `COLONY64_PLAN` (one primary test, frozen before a one-board pilot), `stage64Grid`, `stage64PilotGrid`, `pilotCheck64`, `costRule64`, `stage64Summaries` and `writeStage64`.

In [ ]:
#| default_exp colony64

In [ ]:
#| hide
from nbdev.showdoc import *

## Grid, plan and gates

In [ ]:
#| export
import numpy as np

from explRL.colony_grid import COLONY_SEEDS, PILOT_SEED, colonyCell
from explRL.colony_report import NAN, bootSeeds, cellVals, pairs, signFlipExact, trendSlopes
from explRL.fix_grid import fixEstimate

N64, STAGE64_ARMS, SIZES64 = 64, ('plain', 'colonyGuards'), [8, 16, 32, 64]


def stage64Grid(seeds=COLONY_SEEDS):
    "Study colony64: plain and colonyGuards at N 64 with independent liars (16 cells; no clean boards)."
    return [colonyCell('colony64', a, N64, s, 'independent') for a in STAGE64_ARMS for s in seeds]


def stage64PilotGrid():
    "The pilot (not analysed): colonyGuards with independent liars at N 64, the pilot seed (one board)."
    return [colonyCell('colony64Pilot', 'colonyGuards', N64, PILOT_SEED, 'independent')]


COLONY64_PLAN = dict(
    version=1, frozen=True, deviation=dict(
        date='2026-10-20', approvedBy='user', of='COLONY_PLAN go/no-go (stop at N 16) and COLONY2_PLAN (N 32 and 64, not run)',
        why='guards beat the plain board at 8, 16 and 32 agents with no trend over size; 64 agents widens the range to eightfold '
            'while the Colab accounts train Phase 17', lean='independent liars only, no clean boards: the primary test needs only the liar boards, and the clean cost is already '
                 'measured at 8-32 agents (0.04-0.08)'),
    cells='stage64Grid: plain and colonyGuards x independent liars x N 64 x seeds 1800-1807', cap=250.0,
    capNote='raised from $200 to $250 by the user (2026-10-20); $146.44 spent before this run',
    pilot=dict(cells='colonyGuards|independent, N 64, seed 1890 (not analysed)', maxInvalid=0.05),
    costRule=dict(run=35.0, otherwise='stop and ask', projection="Stage 64's new calls x the pilot's cost per call"),
    primary=dict(H18_12='colonyGuards - plain, honestPooled, independent liars, N 64, 8 seeds paired: one-sided exact sign-flip > 0',
                 alpha=0.05),
    secondary=dict(trend='per seed, the slope over log2 N (8, 16, 32, 64) of the guard advantage with independent liars (Stage 1 '
                         'and the 32-agent rows for 8-32); mean, interval and two-sided sign-flip p, descriptive',
                   cells='honest pooled share, contamination, contests and confirmations per false value, share of turns sharing, '
                         'per cell at N 64'),
    nboot=2000, exploratory=True)


stage64Estimate = lambda cells: fixEstimate(cells, stage64PilotGrid())     # the pilot's board is cached


def pilotCheck64(rows, plan=COLONY64_PLAN):
    "The pilot's invalid share and cost per call, and the cost rule's verdict for Stage 64."
    calls = sum(float(r['calls']) for r in rows) or 1.0
    inv, per = sum(float(r['invalid']) for r in rows) / calls, sum(float(r['cost']) for r in rows) / calls
    return dict(invalid=inv, costPerCall=per, ok=inv < plan['pilot']['maxInvalid'], **costRule64(per, plan))


def costRule64(perCall, plan=COLONY64_PLAN):
    "Run if Stage 64 projects at most plan['costRule']['run'] dollars, else stop and ask."
    proj = stage64Estimate(stage64Grid())['newCalls'] * perCall
    return dict(verdict='run' if proj <= plan['costRule']['run'] else 'stop', projected=proj)

## Analysis

In [ ]:
#| export
def stage64Summaries(rows64, rowsPrior=(), plan=COLONY64_PLAN, nboot=2000):
    "H18-12 (primary), the guard advantage's trend over 8-64 agents (with Stage 1 and the 32-agent rows), per-cell means at N 64."
    dp = pairs(rows64, 'colonyGuards', 'honestPooled', 'independent', sizes=[N64])
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    h = dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(list(dp.values())))
    h['supported'] = bool(h['p'] < plan['primary']['alpha'] and est > 0)
    sl = trendSlopes(list(rowsPrior) + list(rows64), 'colonyGuards', SIZES64)
    p2 = min(1.0, 2 * min(signFlipExact(list(sl.values()), 'greater'), signFlipExact(list(sl.values()), 'less'))) if sl else NAN
    tr = dict(zip(('estimate', 'lo', 'hi'), bootSeeds({(s, 0): v for s, v in sl.items()}, nboot=nboot))) if sl else {}
    allRows = list(rowsPrior) + list(rows64)
    adv = {N: float(np.mean(list(pairs(allRows, 'colonyGuards', 'honestPooled', 'independent', sizes=[N]).values()) or [NAN]))
           for N in SIZES64}
    keys = [k for k in ('honestPooled', 'contamination', 'contestFalse', 'confirmFalse', 'shareShare') if rows64 and k in rows64[0]]
    cells = {f'{a}|independent': {k: float(np.nanmean(list(cellVals(rows64, a, 'independent', k, [N64]).values()) or [NAN]))
                                  for k in keys} for a in STAGE64_ARMS}
    return dict(hyp=dict(H18_12=h), trend=dict(slopes=sl, p=p2, **tr), advantage=adv, cells=cells)


_f = lambda x: 'n/a' if x is None or x != x else f'{x:.3f}'


def writeStage64(path, S, meta):
    "report.md for the 64-agent deviation."
    h, t = S['hyp']['H18_12'], S['trend']
    L = ['# Phase 18: guards at 64 agents (a user-approved deviation; exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| test | n | guards - plain | 95% interval | p (one-sided, exact) | supported |', '|---|---|---|---|---|---|',
         f"| H18-12 independent liars, N 64 (primary) | {h['n']} | {_f(h['estimate'])} | [{_f(h['ci'][0])}, {_f(h['ci'][1])}] | {_f(h['p'])} | {h['supported']} |", '',
         'Guard advantage by team size: ' + ', '.join(f'N {N} {_f(v)}' for N, v in S['advantage'].items()),
         f"Trend over log2 N (8-64): {_f(t.get('estimate'))} per doubling [{_f(t.get('lo'))}, {_f(t.get('hi'))}], two-sided p {_f(t['p'])}", '',
         '| cell | ' + ' | '.join(next(iter(S['cells'].values()))) + ' |', '|---|' + '---|' * len(next(iter(S['cells'].values())))]
    L += [f'| {c} | ' + ' | '.join(_f(v) for v in d.values()) + ' |' for c, d in S['cells'].items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## Tests

In [ ]:
import json, tempfile
from explRL.colony64 import *
from explRL.colony_grid import syntheticRows18
from explRL.coop_grid import coopHash
from explRL.llm_runs import LLMConfig

g = stage64Grid()
assert len(g) == 16 and {c['N'] for c in g} == {64} and len(stage64PilotGrid()) == 1
for c in g:
    cfg = LLMConfig.fromDict(c['cfg'])
    assert cfg.nLiar == 16 and cfg.world.get('guards', 0) == (16 if c['level'].startswith('colonyGuards') else 0) and cfg.world['r'] == 2
assert stage64Estimate(g)['newCalls'] == 16 * 48 * 40 == 30720
assert costRule64(35 / 30720)['verdict'] == 'run' and costRule64(36 / 30720)['verdict'] == 'stop'
pc = pilotCheck64([dict(calls=1920, invalid=19, cost=1.92)])
assert pc['ok'] and abs(pc['costPerCall'] - 0.001) < 1e-12 and pc['verdict'] == 'run' and not pilotCheck64([dict(calls=100, invalid=5, cost=0.1)])['ok']
# analysis on synthetic rows: guards +0.1 at every size (no trend)
rows = [r for r in syntheticRows18({'plain': (0, 0, 0, 0), 'colonyGuards': (0.1, 0, 0, 0)}, sizes=(8, 16, 32, 64), noise=0.0)
        if r['level'].endswith('|independent')]
S = stage64Summaries([r for r in rows if r['N'] == 64], [r for r in rows if r['N'] != 64], nboot=100)
assert abs(S['hyp']['H18_12']['estimate'] - 0.1) < 1e-9 and S['hyp']['H18_12']['p'] == 1 / 256 and S['hyp']['H18_12']['supported']
assert abs(S['trend']['estimate']) < 1e-9 and set(S['advantage']) == {8, 16, 32, 64}
with tempfile.TemporaryDirectory() as d:
    writeStage64(f'{d}/r.md', S, dict(mode='test'))
    assert 'H18-12' in open(f'{d}/r.md').read()
json.dumps(COLONY64_PLAN); assert coopHash(COLONY64_PLAN)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()